# Statistical Machine Learning Rigor & SAM-MFP Feature Prioritization

[![Open in JupyterLab](https://img.shields.io/badge/Launch-JupyterLab-orange.svg)](https://jupyter.org)
[![Python 3.10+](https://img.shields.io/badge/python-3.10+-blue.svg)](https://www.python.org/)

A critical differentiator of MolScope is its mathematical and statistical rigor. This notebook explores:
1. **The Variance Fallacy**: Why ranking molecular features by empirical variance or PCA loadings selects autocorrelated noise.
2. **SAM-MFP Multi-Objective Optimization**: Finding non-dominated features along the 4-criteria Pareto front.
3. **Temporal Data Leakage Prevention**: Purged Block K-Fold Cross-Validation and trajectory-grouped splits.
4. **Supervised ML & LOFO Ranking**: Regularized Random Forests and Leave-One-Feature-Out feature importance.


## 1. The 'Variance Fallacy' in Molecular Dynamics


> **The Variance Fallacy:**
> In molecular dynamics, consecutive simulation frames are strongly temporally correlated. Features with the largest raw sample variance frequently correspond to slow, un-converged fluctuations with Effective Sample Size ratios ($ESS / N < 0.01$) or extreme collinear redundancy ($|r| > 0.95$). Ranking by variance fails to recover true conformational transition drivers.

Let's inspect the benchmark findings from the HP35 Villin Headpiece where variance was compared against SAM-MFP:


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Load benchmark Pareto data
bench_path = Path("benchmarks/sam_mfp/variance_fallacy_rejected_features.csv")
if not bench_path.is_file():
    bench_path = Path("../benchmarks/sam_mfp/variance_fallacy_rejected_features.csv")

if bench_path.is_file():
    rejected_df = pd.read_csv(bench_path)
    print(f"Loaded {len(rejected_df)} high-variance features rejected by SAM-MFP:")
    print(rejected_df[["Feature", "Empirical_Variance", "ESS_Ratio", "Rejection_Reason"]].head(8))
else:
    print("Benchmark tables located in benchmarks/sam_mfp/")


## 2. SAM-MFP Formulation

MolScope formulates feature discovery as a multi-objective Pareto optimization problem:
$$\max \left\{ \text{Representation Salience}, \text{ESS Ratio}, \text{Information Uniqueness}, \text{Replica Consistency} \right\}$$

- **Representation Salience**: Variance captured along PCA / TICA / CKPCA projection manifolds.
- **ESS Ratio ($N_{\text{eff}} / N$)**: Statistical independence accounting for temporal autocorrelation (via initial positive sequence autocorrelation time $\tau_{\text{int}}$).
- **Information Uniqueness**: Orthogonality to already-selected features ($1 - \max |r_s|$).
- **Replica Consistency**: Cross-replica Rank Cosine Similarity (RCS) across independent simulation trajectories.


In [ ]:
# Load Pareto front candidates from real 2RJY benchmark
pareto_path = Path("benchmarks/sam_mfp/real_2rjy_pareto_candidates.csv")
if not pareto_path.is_file():
    pareto_path = Path("../benchmarks/sam_mfp/real_2rjy_pareto_candidates.csv")

if pareto_path.is_file():
    pareto_df = pd.read_csv(pareto_path)
    print(f"Identified {len(pareto_df)} non-dominated Pareto features:")
    print(pareto_df.head(6))


## 3. Visualizing the SAM-MFP Pareto Trade-Off Curve

The plot below shows how MolScope separates genuine conformational drivers from autocorrelated noise:


In [ ]:
# Plot trade-off curve
tradeoff_png = Path("docs/images/sam_mfp_tradeoff.png")

fig, ax = plt.subplots(figsize=(8, 6))

if pareto_path.is_file():
    # If table is available, generate dynamic scatter
    ax.scatter(
        pareto_df["PCA_Salience"],
        pareto_df["ESS_Ratio"],
        c="darkorange",
        s=80,
        edgecolor="black",
        label="SAM-MFP Pareto Front (Optimal)",
    )
    ax.set_xlabel("Target-Free PCA Salience (Representation Power)")
    ax.set_ylabel("Effective Sample Size Ratio (Statistical Independence)")
    ax.set_title("SAM-MFP Multi-Objective Pareto Frontier")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend()
    plt.show()
else:
    print("Run benchmarks/sam_mfp/run_benchmarks.py to regenerate live Pareto data.")


## 4. Preventing Data Leakage in MD Machine Learning

A common pitfall in biophysical machine learning is applying standard $K$-fold cross-validation to time-series trajectories. Consecutive frames $t$ and $t+1$ are near-identical; if $t$ is in train and $t+1$ is in test, models achieve artificially inflated accuracy via interpolation.

MolScope prevents this through:
1. **Purged Block Cross-Validation**: Divides trajectories into contiguous blocks and discards an *embargo window* around test splits.
2. **Trajectory-Grouped Splitting**: Enforces that entire independent simulation replicas are strictly segregated into train or validation folds.


In [ ]:
from molscope.analysis.quality.statistical_rigor import PurgedBlockTimeSeriesSplit

splitter = PurgedBlockTimeSeriesSplit(n_splits=5, embargo_pct=0.02)
print("Purged Block Cross-Validation configured with 2% temporal embargo.")


## 5. Supervised Feature Ranking: Grouped LOFO

MolScope uses **Leave-One-Feature-Out (LOFO)** ablation rather than Gini impurity to rank feature predictive importance:
$$\Delta \text{Score}_j = \text{Score}(\mathcal{F}) - \text{Score}(\mathcal{F} \setminus \{j\})$$
This directly measures the unique predictive value of each coordinate while controlling for collinearity.
